In [13]:
import pandas as pd
from time import perf_counter
from RAG_model.answer.answer import answer
from RAG_model.ingestion.config import BASELINE_RUN_CONFIG, OPENROUTER_BASE_URL

In [14]:
from pathlib import Path

root = Path.cwd()
while not (root / 'data' / 'rag_evaluation' / 'evaluation_questions_v2.csv').exists() and root.parent != root:
    root = root.parent

csv_path = root / 'data' / 'rag_evaluation' / 'evaluation_questions_v2.csv'

questions = pd.read_csv(csv_path)

In [15]:
# Helper Fucntions 

def has_expected_value(retrieved_value, expected_value):
    
    """ 
    Function to see if at least one value from the retrieved return an expected value 
    """
    
    if not expected_value:
        return None
    
    return bool( set(retrieved_value) & set (expected_value))

In [16]:
def reciprocal_rank (retrieved_accession_numbers, expected_accession_number):
    """ 
    Return 1/rank for the first relevant retrieved result.
    Rank starts at 1, return 0 if it was not retrieved
    """
    
    for rank, accesion_number in enumerate(retrieved_accession_numbers,start=1):
        if accesion_number==expected_accession_number:
            return 1/rank

    return 0.0
        

In [17]:
def evaluate_question_deterministic(question_record:dict, run_config:dict):
    
    # Answer the question using the current run_config
    result  = answer(question_record['question'], run_config=run_config ,history=None)
    
    # Get the important values for each chunk
    chunks =  result['Retrieved Chunk texts']
    answer_text = result['Answer']
    cited_chunk_ids = result['Citations']
    
    retrieved_accession_numbers  = [chunk['accession_number'] for chunk in chunks if chunk.get("accession_number")]
    retrieved_chunk_ids = [chunk['chunk_id'] for chunk in chunks if chunk.get("chunk_id")]
    
    expected_accession_number = str(question_record['expected_document']).strip()
    
    if question_record['answerable']:
         # Retrieval: expected SEC filing appears anywhere in top K
        retrieval  = expected_accession_number in retrieved_accession_numbers
        #MRR: Find the rank of the first expected retrieve accession #
        mrr = reciprocal_rank(retrieved_accession_numbers,expected_accession_number)
    else:
        retrieval =  None 
        mrr = None
        
    # Citation validity: Each citation was actually retrieved 
    invalid_citation  = [ citation for citation in cited_chunk_ids if citation not in retrieved_chunk_ids]
    citation_validity = (len(cited_chunk_ids) > 0 and len(invalid_citation) == 0)
    
    # Abstention: only for question not answerable
    if not question_record["answerable"]:
        abstention_correct = (result["Answer"].strip() == "I could not find enough evidence in the retrieved SEC filings.")
    else:
        abstention_correct = None
    
    
    
    values = {
        # Evaluation question
        "question_id": question_record["question_id"],
        "question": question_record["question"],
        "answerable": question_record["answerable"],
        "expected_document": expected_accession_number,
        "reference_answer": question_record["reference_answer"],
        "category": question_record["category"],
        "difficulty": question_record["difficulty"],
        "evaluation_focus": question_record["evaluation_focus"],

        # Experiment identity
        "experiment_name": run_config["experiment_name"],
        "pipeline_version": run_config["pipeline_version"],
        "retrieval_k": run_config["retrieval_k"],
        "embedding_model": run_config["embedding_model"],
        "generation_model": run_config["generation_model"],
        "prompt_version": run_config["prompt_version"],

        # Deterministic retrieval metrics
        "retrieved_chunk_ids": retrieved_chunk_ids,
        "retrieved_chunks" : chunks,
        "retrieved_accession_numbers": retrieved_accession_numbers,
        "similarity_scores": [
            chunk["score"] for chunk in chunks
        ],
        "document_hit_at_k": retrieval,
        "reciprocal_rank": mrr,

        # Deterministic citation metrics
        "citations": cited_chunk_ids,
        "citations_valid": citation_validity,
        "invalid_citations": invalid_citation,

        # Deterministic abstention metric
        "abstention_correct": abstention_correct,

        # Output and operational data
        "answer": answer_text,
        "latency_ms": result["Latency"],
        
        #Token Usage
        "token_usage": result['Token_usage']
        
    }
    
    return values
    
    

### Create a RAGAS function to evaluate 

In [18]:
import os
from openai import AsyncOpenAI
from ragas import SingleTurnSample
from ragas.llms import llm_factory
from ragas.metrics import Faithfulness,FactualCorrectness,LLMContextPrecisionWithReference


C:\Users\david\AppData\Local\Temp\ipykernel_21236\384274377.py:5: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness,FactualCorrectness,LLMContextPrecisionWithReference
C:\Users\david\AppData\Local\Temp\ipykernel_21236\384274377.py:5: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import Faithfulness,FactualCorrectness,LLMContextPrecisionWithReference
C:\Users\david\AppData\Local\Temp\ipykernel_21236\384274377.py:5: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead.

In [19]:
# Create evaluator LLM
ragas_client = AsyncOpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url=OPENROUTER_BASE_URL, timeout=60.0 , max_retries = 5 )
ragas_llm = llm_factory(BASELINE_RUN_CONFIG['ragas_evaluator_model'],provider="openai", client = ragas_client)

# Create metric Objects
faithfulness_metric = Faithfulness(llm=ragas_llm)
factual_correctness_metric = FactualCorrectness(llm=ragas_llm,mode="f1")
context_precision_metric = LLMContextPrecisionWithReference(llm=ragas_llm)

In [20]:
async def evaluate_question_ragas(deterministic_result):

    if not deterministic_result['answerable']:
          return {
                **deterministic_result,
                "ragas_faithfulness" : None,
                "ragas_factual_correctness": None,
                "ragas_context_precision": None
            }
    
    sample  = SingleTurnSample(user_input=deterministic_result['question'], 
                               response=deterministic_result['answer'],
                               reference=deterministic_result['reference_answer'],
                               retrieved_contexts=[chunk['chunk_text'] for chunk in deterministic_result['retrieved_chunks']])
    
    
    faithfulness = await faithfulness_metric.single_turn_ascore(sample)
    factual_correctness = await factual_correctness_metric.single_turn_ascore(sample)
    context_precision = await context_precision_metric.single_turn_ascore(sample)
    
    return {
        **deterministic_result,
        "ragas_faithfulness" : faithfulness,
        "ragas_factual_correctness": factual_correctness,
        "ragas_context_precision": context_precision
    }

## Create a function that retrieves the important metric for each question

In [21]:
async def evaluate_all_questions(questions: pd.DataFrame,run_config):
    question_results = []
    
    # Add a dict for each question with the respective results
    for _ ,row in questions.iterrows():
        question_dict = row.to_dict()
        
        deterministic_answer = evaluate_question_deterministic(question_dict,run_config)
        ragas_result = await evaluate_question_ragas(deterministic_answer)
        
        if ragas_result['answerable']:
            answer_correct = (
                ragas_result['ragas_factual_correctness']>= run_config['answer_correct_threshold']
            )
        else:
            answer_correct=ragas_result['abstention_correct']
        
        question_results.append({
            "question_id": ragas_result["question_id"],
            "category": ragas_result["category"],
            "difficulty": ragas_result["difficulty"],
            "answerable": ragas_result["answerable"],

            "document_hit_at_k": ragas_result["document_hit_at_k"],
            "reciprocal_rank": ragas_result["reciprocal_rank"],
            "citations_valid": ragas_result["citations_valid"],
            "abstention_correct": ragas_result["abstention_correct"],

            "retrieval_latency_ms": ragas_result["latency_ms"]["retrieval"],
            "generation_latency_ms": ragas_result["latency_ms"]["generation"],
            "total_latency_ms": ragas_result["latency_ms"]["total"],
                        
            "ragas_answer_correct":answer_correct,
            "ragas_faithfulness": ragas_result["ragas_faithfulness"],
            "ragas_factual_correctness": (
                ragas_result["ragas_factual_correctness"]
            ),
            "ragas_context_precision": (
                ragas_result["ragas_context_precision"]
            ),
            
            "embedding_tokens": ragas_result["token_usage"]["embedding_total_tokens"],
            "generation_input_tokens": (ragas_result["token_usage"]["generation_input_tokens"]),
            "generation_output_tokens": (ragas_result["token_usage"]["generation_output_tokens"]),
            "rag_total_tokens": ragas_result["token_usage"]["rag_total_tokens"],
        })
        
   
    return {
    "metadata": {
        "experiment_name": run_config["experiment_name"],
        "pipeline_version": run_config["pipeline_version"],
        "retrieval_k": run_config["retrieval_k"],
        "embedding_model": run_config["embedding_model"],
        "generation_model": run_config["generation_model"],
        "prompt_version": run_config["prompt_version"],
    },
   
    "question_results": question_results
    }
    

In [ ]:
questions_metrics = await evaluate_all_questions(questions,BASELINE_RUN_CONFIG)

In [ ]:
df_metrics = pd.DataFrame(questions_metrics['question_results'])
df_metrics.head()

,question_id,category,difficulty,answerable,document_hit_at_k,reciprocal_rank,citations_valid,abstention_correct,latency_ms,ragas_faithfulness,ragas_factual_correctness,ragas_context_precision
0,1,Factual,medium,True,True,0.5,True,None,"{'retrieval': 562.2224000017013, 'prompt_build...",1.00,0.22,0.533333
1,2,Factual,medium,True,False,0.0,True,None,"{'retrieval': 458.99780000036117, 'prompt_buil...",0.00,0.00,0.450000
2,3,Factual,hard,True,True,0.5,True,None,"{'retrieval': 405.1004000011744, 'prompt_build...",1.00,0.55,0.477778
3,4,Factual,medium,True,False,0.0,True,None,"{'retrieval': 477.4892000004911, 'prompt_build...",0.75,0.80,1.000000
4,5,Factual,medium,True,True,1.0,True,None,"{'retrieval': 514.0150000006543, 'prompt_build...",1.00,0.67,1.000000


In [ ]:
def summarize_evaluation(results_df: pd.DataFrame) -> dict:
    """Aggregate one evaluation run into overall and grouped metrics."""

    answerable = results_df[results_df["answerable"]].copy()
    unanswerable = results_df[~results_df["answerable"]].copy()

    overall = {
        "total_questions": len(results_df),
        "answerable_questions": len(answerable),
        "unanswerable_questions": len(unanswerable),

        # Retrieval: only answerable questions have an expected document
        "document_hit_rate_at_k": answerable["document_hit_at_k"].mean(),
        "mrr": answerable["reciprocal_rank"].mean(),

        # Answer outcome: Ragas threshold for answerable; abstention for unanswerable
        "correct_answers": results_df["ragas_answer_correct"].sum(),
        "answer_accuracy": results_df["ragas_answer_correct"].mean(),

        # Grounding / answer quality: answerable questions only
        "citation_validity_rate": results_df["citations_valid"].mean(),
        "mean_ragas_faithfulness": answerable["ragas_faithfulness"].mean(),
        "mean_ragas_factual_correctness": (
            answerable["ragas_factual_correctness"].mean()
        ),
        "mean_ragas_context_precision": (
            answerable["ragas_context_precision"].mean()
        ),

        # Abstention: unanswerable questions only
        "abstention_accuracy": (
            unanswerable["abstention_correct"].mean()
            if not unanswerable.empty
            else None
        ),

        # Efficiency: all questions cost latency/tokens
        "mean_total_latency_ms": results_df["total_latency_ms"].mean(),
        "p95_total_latency_ms": results_df["total_latency_ms"].quantile(0.95),
        "mean_rag_total_tokens": results_df["rag_total_tokens"].mean(),
        "total_rag_tokens": results_df["rag_total_tokens"].sum(),
    }

    by_difficulty = results_df.groupby("difficulty", dropna=False).agg(
        questions=("question_id", "count"),
        correct_answers=("ragas_answer_correct", "sum"),
        answer_accuracy=("ragas_answer_correct", "mean"),
        document_hit_rate_at_k=("document_hit_at_k", "mean"),
        mean_reciprocal_rank=("reciprocal_rank", "mean"),
        mean_factual_correctness=("ragas_factual_correctness", "mean"),
        mean_total_latency_ms=("total_latency_ms", "mean"),
        mean_rag_total_tokens=("rag_total_tokens", "mean"),
    ).reset_index()

    by_category = results_df.groupby("category", dropna=False).agg(
        questions=("question_id", "count"),
        correct_answers=("ragas_answer_correct", "sum"),
        answer_accuracy=("ragas_answer_correct", "mean"),
        document_hit_rate_at_k=("document_hit_at_k", "mean"),
        mean_reciprocal_rank=("reciprocal_rank", "mean"),
        mean_factual_correctness=("ragas_factual_correctness", "mean"),
        mean_total_latency_ms=("total_latency_ms", "mean"),
    ).reset_index()

    return {
        "overall": overall,
        "by_difficulty": by_difficulty,
        "by_category": by_category,
    }

In [ ]:
summary = summarize_evaluation(df_metrics)
summary

KeyError: 'answer_correct_by_ragas'